In [ ]:
%run ./00_Mapping_Setup

In [ ]:
%sql
/* ===================================================================================
   METRIC: EMP05 - Contingent Workers in High-Risk Jurisdictions
   
   BUSINESS QUESTION: 
   Are there contingent workers associated with the Assessable Unit operating in 
   high-risk jurisdictions?
   
   LOGIC SUMMARY:
   1. HIGH-RISK JURISDICTIONS: Queries the td_country_risk_rating_abac table to isolate 
      countries where the FinalABACRating is strictly 'High'.
   2. CONTINGENT WORKERS: Filters the HR Enterprise List (hrbirai_17930...) to find 
      employees where WorkerType contains the word 'contingent'. Cleans the CostCenterID 
      using TRY_CAST to integer to prevent leading-zero mismatch issues.
   3. MAPPING & FLAGS: 
      - Joins the standardized Cost Center Mapping Bootstrap to the HR data.
      - Joins the HR Workcountry to the High-Risk Jurisdictions list.
      - Flags the worker with a 1 if their Workcountry is on the High-Risk list.
   4. OUTPUT FORMATTING: Rolls up the data by AU_ID. If the sum of the High-Risk flag 
      is greater than 0, outputs 'Yes'. Otherwise, safely defaults to 'No'. Matches 
      the A, B, C, D, E column structure of the final reporting template.
=================================================================================== */

WITH High_Risk_Countries AS (
    -- 1. Grab jurisdictions strictly rated as 'High' risk
    SELECT 
        UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Contingent_Workers AS (
    -- 2. Filter HR List for Contingent Workers and prep the Cost Center for joining
    SELECT 
        TRY_CAST(TRIM(CostCenterID) AS INT) AS Cleaned_CC,
        UPPER(TRIM(Workcountry)) AS Workcountry,
        TRIM(WorkerType) AS WorkerType
    FROM hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
    WHERE LOWER(TRIM(WorkerType)) LIKE '%contingent%'
),

Mapped_AUs AS (
    -- 3. Join Mapping -> HR Data -> High Risk Countries
    SELECT 
        m.AU_ID AS BusinessID,
        m.AU_Name,
        -- Check if the employee's country successfully matched the High Risk table
        CASE WHEN h.CountryName IS NOT NULL THEN 1 ELSE 0 END AS Is_High_Risk
    FROM vw_cost_center_mapping_bootstrap m
    
    -- LEFT JOIN to HR data so all AUs appear in the final output
    LEFT JOIN Contingent_Workers c 
        ON CAST(m.Cost_Center_ID AS INT) = c.Cleaned_CC
        
    -- LEFT JOIN to Risk table to flag the high-risk workers
    LEFT JOIN High_Risk_Countries h
        ON c.Workcountry = h.CountryName
)

-- 4. Roll everything up to match the Final Output Template
SELECT 
    BusinessID,                          
    AU_Name,                             
    'EMP05' AS QuestionID,               
    'Applicable' AS Applicability,       
    '' AS ApplicabilityRationale,        
    
    CASE 
        WHEN SUM(Is_High_Risk) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response
    
FROM Mapped_AUs
GROUP BY BusinessID, AU_Name
ORDER BY BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EMP05 - Contingent Worker Drill-Down (High-Risk Only)
   
   PURPOSE: 
   Filters out all the noise. Provides a row-by-row view of ONLY the Contingent Workers 
   that successfully mapped to an Assessable Unit AND are located in a High-Risk country.
   
   LOGIC SUMMARY:
   - Uses an INNER JOIN on the HR data to isolate Contingent Workers.
   - Uses an INNER JOIN on the High-Risk Countries table to strictly return rows 
     that trigger a 'Yes' in the main EMP05 metric.
=================================================================================== */

WITH High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
)

SELECT 
    m.AU_ID AS BusinessID,
    m.AU_Name,
    m.Cost_Center_ID AS Mapped_Cost_Center,
    cw.FullName,
    cw.WorkerType,
    cw.Workcountry AS Raw_Country_From_HR_File,
    '🚨 HIGH RISK MATCH' AS System_Risk_Flag
    
FROM vw_cost_center_mapping_bootstrap m

-- INNER JOIN 1: Only keep rows where a Contingent Worker actually exists
INNER JOIN (
    SELECT 
        TRY_CAST(TRIM(CostCenterID) AS INT) AS Cleaned_CC,
        UPPER(TRIM(Workcountry)) AS Workcountry,
        TRIM(WorkerType) AS WorkerType,
        TRIM(FullName) AS FullName
    FROM hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
    WHERE LOWER(TRIM(WorkerType)) LIKE '%contingent%'
) cw ON CAST(m.Cost_Center_ID AS INT) = cw.Cleaned_CC

-- INNER JOIN 2: Only keep rows where the country is High Risk
INNER JOIN High_Risk_Countries h
    ON cw.Workcountry = h.CountryName
    
ORDER BY m.AU_ID;